# Hello Adapter - Granite Switch with HuggingFace

**Duration:** ~5 min

Minimal example of invoking an **embedded LoRA adapter** inside a **Granite Switch** model, using the HuggingFace backend. This notebook uses the **guardian-core** adapter, which evaluates a message against a safety criterion and returns a structured `yes`/`no` score.

**What you'll learn:**
- How to build a single guardian-core call that scores a user message against a safety criterion and prints a parsed `harmful`/`safe` verdict.
- How to load a composed Granite Switch checkpoint with `transformers`.
- How to activate an adapter by passing `adapter_name=...` to `apply_chat_template`.
- The Guardian prompt protocol - how to frame a criterion so the adapter returns a parseable score.

**Adapters used:** `guardian-core` from the [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) library - a general-purpose safety/risk judge that scores any user-supplied criterion (harm, social bias, jailbreaking, groundedness, ...) as `yes`/`no`.

For the recommended inference path (mellea + vLLM), see [`01_hello_mellea.ipynb`](./01_hello_mellea.ipynb). This notebook intentionally uses HuggingFace to show the underlying control-token mechanics.

## Prerequisites

**1 * A composed Granite Switch checkpoint** with the `guardian-core` adapter. The default `MODEL_PATH` below points at the pre-composed `ibm-granite/granite-switch-4.1-3b-preview` on HuggingFace (drawn from the [IBM Granite 4.1 collection](https://huggingface.co/collections/ibm-granite/granite-41-language-models)). To compose your own checkpoint instead, see [`./04_compose_granite_switch.ipynb`](./04_compose_granite_switch.ipynb) and point `MODEL_PATH` at its output directory.

**2 * Dependencies** (CUDA GPU required):

In [ ]:
%pip install "granite-switch[hf,compose]" jupyter



Full setup details (GPU sizes, HF auth) are in [`../PREREQUISITES.md`](../PREREQUISITES.md).


## 1 * Imports and configuration
Imports are grouped up front so the full dependency set is visible at a glance. `MODEL_PATH` defaults to the pre-composed `ibm-granite/granite-switch-4.1-3b-preview`; override it with a local directory or a different HF repo via the `MODEL_PATH` env var.

In [ ]:
import os
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import granite_switch.hf  # registers the HF backend with AutoModel/AutoConfig

assert torch.cuda.is_available(), "CUDA GPU required"

In [ ]:
# Path or HF repo of a composed Granite Switch checkpoint with guardian-core.
MODEL_PATH = os.environ.get("MODEL_PATH", "ibm-granite/granite-switch-4.1-3b-preview")

print(f"MODEL_PATH: {MODEL_PATH}")

## 2 * Get the model

`MODEL_PATH` already points at a composed checkpoint - either the pre-composed `ibm-granite/granite-switch-4.1-3b-preview` (default) or a local directory you produced via [`./04_compose_granite_switch.ipynb`](./04_compose_granite_switch.ipynb). The `from_pretrained` call below will download it on first use.

## 3 * Load the model
Importing `granite_switch.hf` registers the architecture with `transformers.AutoModelForCausalLM`, so the composed checkpoint loads through the standard HuggingFace API.

In [ ]:
print(f"Loading from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForCausalLM.from_pretrained(MODEL_PATH).eval().to("cuda")

# Inspect which adapters are embedded in this checkpoint.
adapter_names = model.config.adapter_names
print(f"Available adapters: {adapter_names}")

ADAPTER_NAME = "guardian-core" if "guardian-core" in adapter_names else adapter_names[0]
print(f"Using adapter: {ADAPTER_NAME}")

## 4 * Guardian prompt protocol
The `guardian-core` adapter is trained to act as a **judge**: given a `<guardian>` block describing a criterion and a scoring schema, it returns a structured response with a `<score>yes|no</score>` verdict.

Two modes:
- **think** - model reasons step-by-step inside `<think>...</think>` tags before scoring.
- **no-think** - model emits empty `<think></think>` tags and a score immediately (faster, no chain-of-thought).

In [ ]:
GUARDIAN_JUDGE_THINK = (
    "<think>As a judge agent, carefully analyze whether the provided text meets the "
    "given judging criteria. Use all available information, including conversations, "
    "documents, and tools. Reason through the evidence step by step before providing "
    "your score. Wrap your reasoning in <think></think> tags, then provide your "
    "score in <score></score> tags."
)
GUARDIAN_JUDGE_NOTHINK = (
    "<no-think>As a judge agent, assess whether the provided text meets the given "
    "judging criteria using all available information, including conversations, "
    "documents, and tools. Provide your score immediately without explanation. "
    "Output empty <think>\\n</think> tags followed by your score in <score></score> tags."
)


def build_guardian_block(criteria: str, think: bool = False) -> str:
    judge_instruction = GUARDIAN_JUDGE_THINK if think else GUARDIAN_JUDGE_NOTHINK
    return (
        f"<guardian>{judge_instruction}\n\n"
        f"### Criteria: {criteria}\n\n"
        f"### Scoring Schema: If the last assistant's text meets the criteria, "
        f"return 'yes'; otherwise, return 'no'."
    )


print(build_guardian_block("harm", think=False)[:300], "...")

## 5 * Invoke the adapter
This is the key moment: `adapter_name=ADAPTER_NAME` tells `apply_chat_template` to insert the adapter's control token into the prompt. At inference time the Granite Switch model reads that control token and routes the relevant LoRA weights into attention.

In [ ]:
test_message = "Hello, how are you today?"
criteria = (
    "Texts deemed universally harmful are those that pose harm across all "
    "contexts and audiences."
)

messages = [
    {"role": "user", "content": test_message},
    {"role": "user", "content": build_guardian_block(criteria, think=False)},
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    adapter_name=ADAPTER_NAME,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)

adapter_output = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print(f"Input text : {test_message!r}")
print(f"Raw output : {adapter_output!r}")

## 6 * Parse the score
The adapter emits `<score>yes</score>` or `<score>no</score>`. Extract it with a small regex and fall back to substring matching if the tag is malformed.

In [ ]:
def parse_guardian_output(text: str) -> str | None:
    m = re.search(r"<score>\s*(yes|no)\s*</score>", text, re.IGNORECASE)
    if m:
        return m.group(1).lower()
    low = text.lower()
    if "yes" in low:
        return "yes"
    if "no" in low:
        return "no"
    return None


score = parse_guardian_output(adapter_output)
if score is None:
    print(f"WARNING: could not parse score from {adapter_output!r}")
else:
    verdict = "harmful" if score == "yes" else "safe"
    print(f"Guardian verdict: {score!r} -> {verdict}")

## 7 * Next steps

- **Try the Mellea path.** [`01_hello_mellea.ipynb`](./01_hello_mellea.ipynb) runs the same intrinsic through Mellea's wrappers on vLLM - constrained decoding and output parsing come for free.
- **Go deeper on HF mechanics.** [`02_granite_switch_with_hf.ipynb`](./02_granite_switch_with_hf.ipynb) walks through composing a checkpoint and invoking adapters turn-by-turn with the HuggingFace backend.
- **See a full pipeline.** [`03_01_govt_rag_pipeline_simple.ipynb`](./03_01_govt_rag_pipeline_simple.ipynb) chains guardian, query-rewrite, answerability, clarification, and citations into one RAG loop.
- **Compose your own checkpoint.** [`04_compose_granite_switch.ipynb`](./04_compose_granite_switch.ipynb) - pick adapters from the IBM libraries and bake them into a single model.
- **Watch ALORA vs LoRA race.** [`05_alora_vs_lora_race.ipynb`](./05_alora_vs_lora_race.ipynb) compares the two activation styles head-to-head on the same workload.